# 03 - GRPO vs SDPO vs RLSD on an RLM environment

Three-arm comparison proposed in `documentation/rl-method-for-rlm.md`, run on an RLM-style environment (`rlm_rl/rlm_env.py`):

- The root model (Qwen3.8-27B) never sees the long text. It is a `context` variable in a sandboxed Python REPL subprocess.
- Each turn the root writes one Python block. It can call `llm_query` / `llm_batch` (served by the same base model through a local bridge) and ends with `submit(answer)`.
- Tasks: semantic counting over ~200-320 customer messages, and filtered aggregation over ~300-450 order records. Reward: 1 for exact, up to 0.5 partial credit when close, 0 without `submit`.

Arms (identical tasks, seeds, LR and loss settings; only the per-token advantage differs):

| Arm | Per-token advantage |
|---|---|
| `grpo` | `A_i = r_i - mean(r)` on every token |
| `sdpo` | `A_i + tau * delta_t`, tau = 0.11 (lambda = 0.9) |
| `rlsd` | `A_i * ((1 - lam) + lam * clip(exp(sign(A_i) * delta_t), 0.8, 1.2))`, lam = 0.5 |

`delta_t` is the self-teacher's log-ratio: the current checkpoint re-reads the root conversation with a successful sibling's code (answer removed) or the environment's feedback on the failed attempt. Never the gold answer.

Three training steps per arm is enough to check that each method trains stably on real RLM rollouts and to see early trends. It is not enough to rank them with confidence; see the noise estimate at the end.

In [1]:
import asyncio, json, os, sys, time
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")  # provides WANDB_API_KEY
assert os.environ.get("WANDB_API_KEY"), "WANDB_API_KEY missing from .env"

import art
from art.serverless.backend import ServerlessBackend

from rlm_rl.rlm_env import SubcallBridge, evaluate_rlm, gather_rlm_groups, make_rlm_tasks
from rlm_rl.shaping import shape_groups, train_kwargs

In [2]:
BASE_MODEL = "Qwen/Qwen3.8-27B"
PROJECT = "rlm-three-way"
RUN_TAG = "v1"
ARMS = {"grpo": None, "sdpo": 0.11, "rlsd": 0.1}  # arm -> tau for the advantage channel
TRAIN_STEPS = 3
TASKS_PER_STEP = 8
ROLLOUTS_PER_TASK = 4
LEARNING_RATE = 1e-5
EVAL_TASKS = make_rlm_tasks(24, seed=9999)
EVAL_SAMPLES = 2
RESULTS_PATH = ROOT / "results" / "three_way_rlm.json"
RESULTS_PATH.parent.mkdir(exist_ok=True)

backend = ServerlessBackend()
models = {}
for arm in ARMS:
    models[arm] = art.TrainableModel(
        name=f"qwen3-8-27b-{arm}", project=PROJECT, run_name=f"qwen3-8-27b-{arm}-{RUN_TAG}", base_model=BASE_MODEL
    )
    await models[arm].register(backend)

# Sub-calls always go to the untrained base model (step 0 is identical for every arm).
anchor = models["grpo"]
bridge = SubcallBridge(anchor.inference_base_url, anchor.inference_api_key, anchor.get_inference_name(step=0))

results = {"config": {"base_model": BASE_MODEL, "train_steps": TRAIN_STEPS, "tasks_per_step": TASKS_PER_STEP,
                      "rollouts_per_task": ROLLOUTS_PER_TASK, "learning_rate": LEARNING_RATE,
                      "eval_tasks": len(EVAL_TASKS), "eval_samples": EVAL_SAMPLES, "arms": ARMS},
           "steps": {arm: [] for arm in ARMS}, "eval": {}}

def save():
    RESULTS_PATH.write_text(json.dumps(results, indent=2))

## Baseline: the untrained model on the held-out tasks

In [3]:
results["eval"]["base"] = await evaluate_rlm(anchor, 0, EVAL_TASKS, bridge, EVAL_SAMPLES)
save()
results["eval"]["base"]

{'accuracy': 0.3541666666666667,
 'mean_reward': 0.4948729166666667,
 'submitted': 0.7708333333333334,
 'turns': 5.229166666666667,
 'subcalls': 0.6875,
 'first_turn_submit': 0.041666666666666664,
 'n': 48,
 'failed_episodes': 0}

## Train three steps per arm

All arms see the same task set at each step (seeded by step), and run concurrently.

In [4]:
def group_stats(groups):
    trajs = [t for g in groups for t in g.trajectories]
    mean = lambda k: sum(t.metrics[k] for t in trajs) / len(trajs)
    mixed = sum(1 for g in groups if len({t.reward for t in g.trajectories}) > 1)
    return {"train_reward": sum(t.reward for t in trajs) / len(trajs), "train_correct": mean("correct"),
            "turns": mean("turns"), "subcalls": mean("subcalls"), "used_subcalls": mean("used_subcalls"),
            "first_turn_submit": mean("first_turn_submit"), "submitted": mean("submitted"),
            "mixed_groups": mixed, "groups": len(groups), "trajectories": len(trajs)}


async def run_arm(arm: str, tau):
    model, step = models[arm], 0
    for i in range(TRAIN_STEPS):
        started = time.time()
        tasks = make_rlm_tasks(TASKS_PER_STEP, seed=1000 + i)
        groups = await gather_rlm_groups(model, step, tasks, ROLLOUTS_PER_TASK, bridge)
        row = {"step": step, **group_stats(groups)}
        await model.log(groups, split="train", step=step)
        if arm != "grpo":
            groups, stats = await shape_groups(model, step, groups, arm, tau=tau, seed=i)
            row["shaping"] = stats.summary()
        result = await backend.train(model, groups, learning_rate=LEARNING_RATE, **train_kwargs(arm, tau))
        await model.log(metrics=result.metrics, step=result.step)
        step = result.step
        row.update(trained_to=step, seconds=round(time.time() - started),
                   loss={k: v for k, v in result.metrics.items() if k.startswith("loss/")})
        results["steps"][arm].append(row)
        save()
        print(f"[{arm}] step {i + 1}/{TRAIN_STEPS}: reward={row['train_reward']:.3f} mixed_groups={row['mixed_groups']}/{row['groups']}")
    results["eval"][arm] = await evaluate_rlm(model, step, EVAL_TASKS, bridge, EVAL_SAMPLES)
    save()


await asyncio.gather(*(run_arm(arm, tau) for arm, tau in ARMS.items()))

qwen3-8-27b-grpo@step0:   0%|          | 0/32 [00:00<?, ?it/s]

qwen3-8-27b-sdpo@step0:   0%|          | 0/32 [00:00<?, ?it/s]

qwen3-8-27b-rlsd@step0:   0%|          | 0/32 [00:00<?, ?it/s]

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: rishi-jain1 (models-ontario-tech-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

weave: 🍩 https://wandb.ai/models-ontario-tech-university/rlm-three-way/r/call/01a124b5-e8fa-7865-a08e-7eefeef5eb01


weave: 🍩 https://wandb.ai/models-ontario-tech-university/rlm-three-way/r/call/01a124b5-e0af-7f4e-af3c-26480a03a9bd


weave: 🍩 https://wandb.ai/models-ontario-tech-university/rlm-three-way/r/call/01a124b5-e8f9-7557-aab9-e4a249d35e0e


weave: 🍩 https://wandb.ai/models-ontario-tech-university/rlm-three-way/r/call/01a124b5-f84c-7bde-ab53-28329955c860


[None, None, None]

## Results

In [5]:
print(f"{'':10}{'eval acc':>10}{'eval reward':>13}{'turns':>8}{'subcalls':>10}{'1st-turn submit':>17}")
for name in ["base", *ARMS]:
    e = results["eval"][name]
    print(f"{name:10}{e['accuracy']:>10.3f}{e['mean_reward']:>13.3f}{e['turns']:>8.2f}{e['subcalls']:>10.2f}{e['first_turn_submit']:>17.3f}")

print("\nTraining reward by step (rollouts sampled from that step's checkpoint)")
for arm in ARMS:
    print(f"{arm:6}", "  ".join(f"{r['train_reward']:.3f}" for r in results["steps"][arm]))

n = results["eval"]["base"]["n"]
p = results["eval"]["base"]["accuracy"]
print(f"\nNoise: with n={n} eval episodes, one standard error of accuracy is about {((p * (1 - p)) / n) ** 0.5:.3f}")

            eval acc  eval reward   turns  subcalls  1st-turn submit
base           0.354        0.495    5.23      0.69            0.042
grpo           0.458        0.585    5.31      1.27            0.000
sdpo           0.375        0.538    5.27      0.60            0.000
rlsd           0.542        0.599    5.25      0.96            0.000

Training reward by step (rollouts sampled from that step's checkpoint)
grpo   0.513  0.553  0.524
sdpo   0.438  0.499  0.454
rlsd   0.520  0.524  0.593

Noise: with n=48 eval episodes, one standard error of accuracy is about 0.069
